# Day 1 — Handling Missing Data
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

Real-world datasets always have missing values. This notebook covers:
- Understanding the three types of missingness (MCAR / MAR / MNAR)
- Profiling and visualising missing data patterns
- Simple imputation strategies (mean, median, mode)
- Advanced imputation (KNN, MICE/IterativeImputer)
- Missing indicator features
- Leakage-safe imputation inside a Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
print('Libraries loaded successfully.')

## 1. Create a Realistic Dataset with Controlled Missingness

In [ ]:
# Load California housing and inject controlled missing values
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

# Inject MCAR: randomly drop 8% of MedInc
mcar_mask = np.random.rand(len(df)) < 0.08
df.loc[mcar_mask, 'MedInc'] = np.nan

# Inject MAR: drop AveRooms when MedInc (original) is > 4 (higher income = less likely to report)
mar_mask = (data.frame['MedInc'] > 4) & (np.random.rand(len(df)) < 0.15)
df.loc[mar_mask, 'AveRooms'] = np.nan

# Inject MNAR: drop AveOccup for very high values (high occupancy = people don't report)
mnar_mask = (data.frame['AveOccup'] > 3.0) & (np.random.rand(len(df)) < 0.20)
df.loc[mnar_mask, 'AveOccup'] = np.nan

print(df.shape)
df.head()

## 2. Profile Missingness

In [ ]:
# Missing data report
missing_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2)
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print('=== Missing Data Report ===')
print(missing_report)

# Visualise missing patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of missing %
missing_report['missing_pct'].plot(kind='bar', ax=axes[0], color='#ef4444', alpha=0.8)
axes[0].set_title('Missing Data Percentage per Feature', fontsize=13)
axes[0].set_ylabel('Missing (%)')
axes[0].set_xlabel('')
axes[0].axhline(y=5, color='orange', linestyle='--', label='5% threshold')
axes[0].legend()

# Heatmap of missing pattern (sample)
sample = df.isnull().astype(int).sample(200, random_state=42)
sns.heatmap(sample[missing_report.index], cmap=['#1e293b', '#ef4444'],
            ax=axes[1], cbar=False, yticklabels=False)
axes[1].set_title('Missing Pattern (200 rows sample)', fontsize=13)

plt.tight_layout()
plt.show()

## 3. Check for Correlated Missingness (MAR / MNAR signal)

In [ ]:
# Create binary missing indicators
for col in ['MedInc', 'AveRooms', 'AveOccup']:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

# Correlation between missingness flags
missing_cols = [c for c in df.columns if c.endswith('_missing')]
corr = df[missing_cols].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5)
plt.title('Correlation Between Missingness Flags')
plt.tight_layout()
plt.show()

# Drop helper columns for now
df.drop(columns=missing_cols, inplace=True)
print('If two flags are highly correlated, both features go missing together → MNAR signal.')

## 4. Train/Test Split FIRST (Golden Rule)

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Rule: ALL imputers are fit on X_train ONLY.')

## 5. Simple Imputation Strategies

In [ ]:
# Mean imputation
mean_imp = SimpleImputer(strategy='mean')
X_train_mean = mean_imp.fit_transform(X_train)
X_test_mean  = mean_imp.transform(X_test)
print('Mean imputer statistics (first 3 features):',
      mean_imp.statistics_[:3].round(3))

# Median imputation (preferred for skewed distributions)
median_imp = SimpleImputer(strategy='median')
X_train_median = median_imp.fit_transform(X_train)

# Mode imputation for categorical (demo with a string column)
cat_demo = pd.DataFrame({'city': ['London', 'Paris', np.nan, 'London', np.nan, 'Berlin']})
mode_imp = SimpleImputer(strategy='most_frequent')
cat_imputed = mode_imp.fit_transform(cat_demo)
print('\nMode imputation on categorical:')
print(pd.DataFrame(cat_imputed, columns=['city']))

## 6. Missing Indicator Features

In [ ]:
# add_indicator=True appends binary flag columns
indicator_imp = SimpleImputer(strategy='median', add_indicator=True)
X_train_ind = indicator_imp.fit_transform(X_train)
print(f'Shape before: {X_train.shape}')
print(f'Shape after (with indicators): {X_train_ind.shape}')
print('Extra columns = one binary flag per feature that had missing values.')

## 7. KNN Imputation — Captures Feature Correlations

In [ ]:
# KNNImputer fills missing with mean of k nearest complete neighbours
# Note: scale first (KNN is distance-based)
from sklearn.preprocessing import StandardScaler

knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn_imp', KNNImputer(n_neighbors=5)),
])

X_train_knn = knn_pipe.fit_transform(X_train)
X_test_knn  = knn_pipe.transform(X_test)
print(f'KNN imputed shape: {X_train_knn.shape}')
print('KNN imputation requires no NaNs remain:', not np.any(np.isnan(X_train_knn)))

## 8. Iterative Imputation (MICE)

MICE (Multiple Imputation by Chained Equations) models each feature as a function of all others — iteratively refines imputed values.

In [ ]:
mice_imp = IterativeImputer(max_iter=10, random_state=42, verbose=0)
X_train_mice = mice_imp.fit_transform(X_train)
X_test_mice  = mice_imp.transform(X_test)
print(f'MICE imputed shape: {X_train_mice.shape}')
print('No missing values remain:', not np.any(np.isnan(X_train_mice)))

## 9. Compare Imputation Strategies on a Target Column

Evaluate which imputation method best preserves the original distribution of `MedInc`.

In [ ]:
# Compare distributions for MedInc (index 0)
medinc_original = data.frame['MedInc'].values

imp_results = {
    'Original (no missing)': medinc_original,
    'Mean Imputed': X_train_mean[:, 0],
    'Median Imputed': X_train_median[:, 0],
    'MICE Imputed': X_train_mice[:, 0],
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
colors = ['#22c55e', '#3b82f6', '#f59e0b', '#a78bfa']

for ax, (label, vals), color in zip(axes, imp_results.items(), colors):
    ax.hist(vals, bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('MedInc')

plt.suptitle('Imputation Strategy Comparison — MedInc Distribution', fontsize=12)
plt.tight_layout()
plt.show()
print('Mean imputation creates a spike at the mean. Median slightly better. MICE preserves distribution best.')

## 10. Production-Ready Imputation Pipeline

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Full pipeline: impute → scale → model
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'RMSE with median+indicator pipeline: {rmse:.4f}')
print('Pipeline is ready for deployment — joblib.dump(pipeline, "model.pkl")')

## Exercises

1. **Missingness detection**: Load the Titanic dataset (`sns.load_dataset('titanic')`). Profile missing values and identify which mechanism (MCAR/MAR/MNAR) best describes each missing column.

2. **Imputation comparison**: Compare mean vs median vs KNN imputation on the `age` column. Plot the resulting distributions and compute the mean absolute difference from the original (before you introduced missingness).

3. **Leakage demo**: Deliberately fit a SimpleImputer on the full dataset (before split), then compare CV scores to a correctly-built Pipeline. Quantify the optimism introduced by leakage.

4. **MICE vs KNN**: On a dataset with correlated features (create synthetic: 5 correlated features), compare MICE vs KNN imputation quality using RMSE on held-out values.

In [ ]:
# ── Exercise 1 Solution: Titanic missing data profile ──
titanic = sns.load_dataset('titanic')
print('=== Titanic Missing Data ===')
missing = pd.DataFrame({
    'missing_count': titanic.isnull().sum(),
    'missing_pct': (titanic.isnull().mean() * 100).round(2)
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)
print(missing)
print("\nage: ~20% missing — likely MAR (younger/older passengers less likely to have age recorded)")
print("deck: ~77% missing — likely MNAR (deck info systematically absent for lower classes)")
print("embark_town: 2 missing — likely MCAR (random data entry errors)")